# 01. Adversarial Robustness: FGSM, PGD, TRADES, and Randomized Smoothing

- **Category:** 11_ai_safety_privacy_and_verification
- **Lineage & Citations:** Goodfellow, Shlens, & Szegedy (2014 *Explaining and harnessing adversarial examples*); Madry et al. (2018 *Towards Deep Learning Models Resistant to Adversarial Attacks*); Zhang et al. (2019 *Theoretically Principled Trade-off between Robustness and Accuracy: TRADES*); Cohen, Rosenfeld, & Kolter (2019 *Certified Adversarial Robustness via Randomized Smoothing*).
- **Purpose:** Formulate and evaluate adversarial attacks and defenses: Fast Gradient Sign Method (FGSM), multi-step Projected Gradient Descent (PGD) in $L_\infty$ balls, TRADES robust regularized training, and certified $L_2$ radii via Randomized Smoothing.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [12. Topological Data Analysis Persistent Homology and Singular Learning LLC](../10_xai_and_ai_analysis_methods/12_topological_data_analysis_persistent_homology_and_singular_learning_llc.ipynb) $\to$ `🔗 Downstream Syntheses:` [02. Formal Neural Verification IBP and CROWN Bound Propagation](02_formal_neural_verification_ibp_and_crown_bound_propagation.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do not rely on empirical 1-step FGSM resistance as a guarantee of robustness (obfuscated and shattered gradients create false security against multi-step PGD); do not apply standard empirical defenses without verified bounds in high-assurance or safety-critical domains.

## Mathematical Foundations & Governing Equations

1. **FGSM Perturbation**: $x_{\text{adv}} = \text{clamp}(x + \epsilon \cdot \text{sign}(\nabla_x \mathcal{L}), x_{\min}, x_{\max})$
2. **PGD Step in $L_\infty$**: $x^{(t+1)} = \text{clamp}(x^{(t)} + \alpha \cdot \text{sign}(\nabla_{x^{(t)}} \mathcal{L}), x - \epsilon, x + \epsilon)$
3. **TRADES Regularized Loss**: $\mathcal{L} = \mathcal{L}_{\text{CE}}(f(x), y) + \beta \cdot D_{\text{KL}}(P(f(x)) \parallel P(f(x_{\text{adv}})))$
4. **Randomized Smoothing Radius**: $R = \frac{\sigma}{2} (\Phi^{-1}(p_{\text{top}}) - \Phi^{-1}(p_{\text{runnerup}}))$

In [ ]:
import sys, time
from pathlib import Path
import torch
from safetensors.torch import load_file

r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
if str(r) not in sys.path: sys.path.insert(0, str(r))
w_path = r / "weights/11_ai_safety_privacy_and_verification/01_adversarial_robustness_fgsm_pgd_trades_gcg_and_smoothing.safetensors"
weights = load_file(str(w_path))
print(f"Loaded {len(weights)} tensors from {w_path.name}")

In [ ]:
def calc_fgsm(x, grad, eps=0.1, x_min=0.0, x_max=1.0):
    return torch.clamp(x + float(eps) * torch.sign(grad), float(x_min), float(x_max))

def calc_pgd_step(x_adv, x_orig, grad, alpha=0.04, eps=0.1, x_min=0.0, x_max=1.0):
    step = x_adv + float(alpha) * torch.sign(grad)
    proj = torch.clamp(step, x_orig - float(eps), x_orig + float(eps))
    return torch.clamp(proj, float(x_min), float(x_max))

def calc_trades_loss(cl_logits, adv_logits, labels, beta=1.0):
    ce = torch.nn.functional.cross_entropy(cl_logits, labels.long())
    p_cl = torch.softmax(cl_logits, dim=-1)
    log_p_cl = torch.log_softmax(cl_logits, dim=-1)
    log_p_adv = torch.log_softmax(adv_logits, dim=-1)
    kl = torch.sum(p_cl * (log_p_cl - log_p_adv), dim=-1).mean()
    return ce + float(beta) * kl

def calc_smoothing_radius(p_top, p_run, sigma=0.25):
    sqrt2 = 1.4142135623730951
    z_top = sqrt2 * torch.erfinv(2.0 * p_top - 1.0)
    z_run = sqrt2 * torch.erfinv(2.0 * p_run - 1.0)
    return (float(sigma) / 2.0) * (z_top - z_run)

In [ ]:
# Parity verification against golden reference tensors
fgsm = calc_fgsm(weights["adv_x"], weights["adv_grad"], 0.1, 0.0, 1.0)
assert torch.max(torch.abs(fgsm - weights["fgsm_adv_ref"])) < 1e-5, "FGSM mismatch"

pgd = calc_pgd_step(fgsm, weights["adv_x"], weights["adv_grad"], 0.04, 0.1, 0.0, 1.0)
assert torch.max(torch.abs(pgd - weights["pgd_step_ref"])) < 1e-5, "PGD mismatch"

t_loss = calc_trades_loss(weights["clean_logits"], weights["adv_logits"], weights["labels"], 1.0)
assert torch.max(torch.abs(t_loss - weights["trades_loss_ref"])) < 1e-5, "TRADES mismatch"

radius = calc_smoothing_radius(weights["p_top"], weights["p_runnerup"], 0.25)
assert torch.max(torch.abs(radius - weights["smoothing_radius_ref"])) < 1e-5, "Radius mismatch"
print("All FGSM, PGD, TRADES, and Randomized Smoothing assertions passed!")

In [ ]:
# Benchmarking robustness operators
t0 = time.perf_counter()
for _ in range(500):
    adv = calc_fgsm(weights["adv_x"], weights["adv_grad"], 0.1, 0.0, 1.0)
    _ = calc_pgd_step(adv, weights["adv_x"], weights["adv_grad"], 0.04, 0.1, 0.0, 1.0)
    _ = calc_trades_loss(weights["clean_logits"], weights["adv_logits"], weights["labels"], 1.0)
    _ = calc_smoothing_radius(weights["p_top"], weights["p_runnerup"], 0.25)
lat_us = (time.perf_counter() - t0) / 500 * 1e6
print(f"Fused FGSM + PGD + TRADES + Smoothing latency: {lat_us:.2f} us")

In [ ]:
# Failure Mode: Clean vs Adversarial Decision Flip under Undefended Model
w_class = torch.tensor([[2.0], [-3.0], [1.5]])  # Decision boundary weight vector
x_clean = torch.tensor([[0.5, 0.2, 0.8]])
margin_clean = (x_clean @ w_class).item()  # Clean output margin = 0.5*2 - 0.2*3 + 0.8*1.5 = 1.0 - 0.6 + 1.2 = +1.6

# Adversary targets negative class by perturbing against the gradient direction (grad = w_class)
grad_x = w_class.t()  # [1, 3]
# Perturb with epsilon = 0.35 in opposite direction to flip prediction
x_adv = torch.clamp(x_clean - 0.35 * torch.sign(grad_x), 0.0, 1.0)
margin_adv = (x_adv @ w_class).item()
print(f"Clean margin: {margin_clean:.3f} (Correct class) -> Adversarial margin: {margin_adv:.3f} (Flipped!)")
assert margin_clean > 1.0, "Clean sample must be classified with high positive confidence!"
assert margin_adv < 0.0, "Small bounded perturbation flips prediction on undefended model!"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **FGSM and PGD** exploit local linear sensitivity in neural networks to find worst-case $L_\infty$ perturbations.
2. **TRADES** balances clean classification risk with decision boundary stability via output KL divergence regularization.
3. **Randomized Smoothing** guarantees provable, certifiable $L_2$ robustness radii using isotropic Gaussian noise and the Neyman-Pearson lemma.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [01_adversarial_robustness_fgsm_pgd_trades_gcg_and_smoothing.md](../../markdown/11_ai_safety_privacy_and_verification/01_adversarial_robustness_fgsm_pgd_trades_gcg_and_smoothing.md)
- **PyTorch 2.x Module:** [01_adversarial_robustness_fgsm_pgd_trades_gcg_and_smoothing.py](../../pytorch2/11_ai_safety_privacy_and_verification/01_adversarial_robustness_fgsm_pgd_trades_gcg_and_smoothing.py)
- **Functional JAX Module:** [01_adversarial_robustness_fgsm_pgd_trades_gcg_and_smoothing.py](../../jax/11_ai_safety_privacy_and_verification/01_adversarial_robustness_fgsm_pgd_trades_gcg_and_smoothing.py)
- **Native Rust Candle Track:** [01_adversarial_robustness_fgsm_pgd_trades_gcg_and_smoothing.rs](../../candle/11_ai_safety_privacy_and_verification/01_adversarial_robustness_fgsm_pgd_trades_gcg_and_smoothing.rs)
- **Hardware Kernel (CPU / WGSL):** [01_adversarial_robustness_fgsm_pgd_trades_gcg_and_smoothing.rs](../../kernels/11_ai_safety_privacy_and_verification/01_adversarial_robustness_fgsm_pgd_trades_gcg_and_smoothing.rs)
- **Golden SafeTensors:** [01_adversarial_robustness_fgsm_pgd_trades_gcg_and_smoothing.safetensors](../../weights/11_ai_safety_privacy_and_verification/01_adversarial_robustness_fgsm_pgd_trades_gcg_and_smoothing.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 12. Topological Data Analysis Persistent Homology and Singular Learning LLC](../10_xai_and_ai_analysis_methods/12_topological_data_analysis_persistent_homology_and_singular_learning_llc.ipynb) | [Master Index](../../README.md#11_ai_safety_privacy_and_verification) | [Next: 02. Formal Neural Verification IBP and CROWN Bound Propagation](02_formal_neural_verification_ibp_and_crown_bound_propagation.ipynb) &rarr;